**Sample ID**: 307




**Query**: Update the Jira issue with summary "Sprint 12 Retrospective" in the project AGILE_TEAM and set status to In Progress and reschedule the Google Calendar event "Sprint 12 Retrospective" in "Work" calendar to April 6, 2025, at 11 AM.




**DB Type**: Edge Case 2




**Case Description**: The status of Jira issue is already set to In Progress, but the calendar event still needs to be rescheduled




**Global/Context Variables**:


- JIRA_ISSUE_SUMMARY = "Sprint 12 Retrospective"
- JIRA_PROJECT_NAME = "AGILE_TEAM"
- ISSUE_NEW_STATUS = "In Progress"
- CALENDAR_EVENT_NAME = "Sprint 12 Retrospective"
- CALENDAR_NAME = "Work"
- CALENDAR_EVENT_NEW_DATE = "2025-04-06T11:00:00Z"






**Datetime Context Variables**:
- user_timezone = "UTC"

**APIs**:

- jira
- google_calendar


# Set Up

## Download relevant files

In [ ]:
import io
import os
import sys
import zipfile
import shutil
import re
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

# Version to download
VERSION = "0.1.5"  # Version of the API

# Define paths
CONTENT_DIR = '/content'
APIS_DIR = os.path.join(CONTENT_DIR, 'APIs')
DBS_DIR = os.path.join(CONTENT_DIR, 'DBs')
SCRIPTS_DIR = os.path.join(CONTENT_DIR, 'Scripts')
FC_DIR = os.path.join(CONTENT_DIR, 'Schemas')
ZIP_PATH = os.path.join(CONTENT_DIR, f'APIs_V{VERSION}.zip')

# Google Drive Folder ID where versioned APIs zip files are stored
APIS_FOLDER_ID = '1QpkAZxXhVFzIbm8qPGPRP1YqXEvJ4uD4'

# List of items to extract from the zip file
ITEMS_TO_EXTRACT = ['APIs/', 'DBs/', 'Scripts/', 'Schemas/']

# Clean up existing directories and files
for path in [APIS_DIR, DBS_DIR, SCRIPTS_DIR, FC_DIR, ZIP_PATH]:
    if os.path.exists(path):
        if os.path.isdir(path):
            shutil.rmtree(path)
        else:
            os.remove(path)

# Authenticate and create the drive service
auth.authenticate_user()
drive_service = build('drive', 'v3')

# Helper function to download a file from Google Drive
def download_drive_file(service, file_id, output_path, file_name=None, show_progress=True):
    """Downloads a file from Google Drive"""
    destination = output_path
    request = service.files().get_media(fileId=file_id)
    with io.FileIO(destination, 'wb') as fh:
        downloader = MediaIoBaseDownload(fh, request)
        done = False
        while not done:
            status, done = downloader.next_chunk()
            if show_progress:
                print(f"Download progress: {int(status.progress() * 100)}%")


# 1. List files in the specified APIs folder
print(f"Searching for APIs zip file with version {VERSION} in folder: {APIS_FOLDER_ID}...")
apis_file_id = None

try:
    query = f"'{APIS_FOLDER_ID}' in parents and trashed=false"
    results = drive_service.files().list(q=query, fields="files(id, name)").execute()
    files = results.get('files', [])
    for file in files:
        file_name = file.get('name', '')
        if file_name.lower() == f'apis_v{VERSION.lower()}.zip':
            apis_file_id = file.get('id')
            print(f"Found matching file: {file_name} (ID: {apis_file_id})")
            break

except Exception as e:
    print(f"An error occurred while listing files in Google Drive: {e}")

if not apis_file_id:
    print(f"Error: Could not find APIs zip file with version {VERSION} in the specified folder.")
    sys.exit("Required APIs zip file not found.")

# 2. Download the found APIs zip file
print(f"Downloading APIs zip file with ID: {apis_file_id}...")
download_drive_file(drive_service, apis_file_id, ZIP_PATH, file_name=f'APIs_V{VERSION}.zip')

# 3. Extract specific items from the zip file to /content
print(f"Extracting specific items from {ZIP_PATH} to {CONTENT_DIR}...")
try:
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_contents = zip_ref.namelist()

        for member in zip_contents:
            extracted = False
            for item_prefix in ITEMS_TO_EXTRACT:
              if member == item_prefix or member.startswith(item_prefix):
                    zip_ref.extract(member, CONTENT_DIR)
                    extracted = True
                    break

except zipfile.BadZipFile:
    print(f"Error: The downloaded file at {ZIP_PATH} is not a valid zip file.")
    sys.exit("Invalid zip file downloaded.")
except Exception as e:
    print(f"An error occurred during extraction: {e}")
    sys.exit("Extraction failed.")


# 4. Clean up
if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)

# 5. Add APIs to path
if os.path.exists(APIS_DIR):
    sys.path.append(APIS_DIR)
else:
    print(f"Error: APIS directory not found at {APIS_DIR} after extraction. Cannot add to path.")

# 6. Quick verification
# Check for the presence of the extracted items
verification_paths = [APIS_DIR, DBS_DIR, SCRIPTS_DIR]
all_present = True
print("\nVerifying extracted items:")
for path in verification_paths:
    if os.path.exists(path):
        print(f"? {path} is present.")
    else:
        print(f"? {path} is MISSING!")
        all_present = False

if all_present:
    print(f"\n? Setup complete! Required items extracted to {CONTENT_DIR}.")
else:
    print("\n? Setup failed! Not all required items were extracted.")

# 7. Generate Schemas

print("\nGenerating FC Schemas")

# Change working directory to the source folder

# Iterate through the packages in the /content/APIs directory

    # Check if it's a directory (to avoid processing files)
        # Call the function to generate schema for the current package
print(f"? Successfully generated {len(os.listdir(FC_DIR))} FC Schemas to {FC_DIR}")
os.chdir(CONTENT_DIR)

Searching for APIs zip file with version 0.1.5 in folder: 1QpkAZxXhVFzIbm8qPGPRP1YqXEvJ4uD4...
Found matching file: APIs_V0.1.5.zip (ID: 1Hkl0_1M8feI6eGcpJrpp5hd_7ITC1iIh)
Download progress: 100%
Extracting specific items from /content/APIs_V0.1.5.zip to /content...

Verifying extracted items:
? /content/APIs is present.
? /content/DBs is present.
? /content/Scripts is present.

? Setup complete! Required items extracted to /content.

Generating FC Schemas
? Successfully generated 73 FC Schemas to /content/Schemas


## Install Dependencies and Clone Repositories

In [ ]:
!pip install -r /content/APIs/requirements.txt

## Import APIs and initiate DBs

In [ ]:
import google_calendar
import jira

# Load the databases
google_calendar.SimulationEngine.db.load_state("/content/DBs/CalendarDefaultDB.json")
jira.SimulationEngine.db.load_state("/content/DBs/JiraDefaultDB.json")

# Constants
JIRA_ISSUE_SUMMARY = "Sprint 12 Retrospective"
JIRA_PROJECT_NAME = "AGILE_TEAM"
ISSUE_NEW_STATUS = "In Progress"
CALENDAR_EVENT_NAME = "Sprint 12 Retrospective"
CALENDAR_NAME = "Work"
CALENDAR_EVENT_NEW_DATE = "2025-04-06T11:00:00Z"



# Variables
JIRA_PROJECT_KEY = "AGT"
JIRA_ISSUE_TYPE="Epic"
CALENDAR_EVENT_LOCATION = "Conference Room A"
CALENDAR_EVENT_DESCRIPTION = "Discussion about Sprint 12."
CALENDAR_EVENT_START_DATETIME = "2025-03-31T11:00:00Z"
CALENDAR_EVENT_END_DATETIME = "2025-03-31T12:00:00Z"


# Create Jira project
project = jira.create_project(proj_key = JIRA_PROJECT_KEY,
                                                      proj_name = JIRA_PROJECT_NAME)

print(f"Project created: {project}")

# Create jira issue with status set to "In Progress" to match the case description
issue = jira.create_issue({
    "project": JIRA_PROJECT_KEY,
    "summary": JIRA_ISSUE_SUMMARY,
    "status": ISSUE_NEW_STATUS,
    "issueType": JIRA_ISSUE_TYPE
})
print(f"Issue created: {issue}")

# Create calendar
new_calendar = google_calendar.create_secondary_calendar(resource={
    "summary": CALENDAR_NAME
})

# Get the calendar id
calendar_id = new_calendar.get("id")
print(f"Calendar created: {new_calendar}")

# Add the calendar to calendar list
google_calendar.create_calendar_list_entry(resource={
    "id": calendar_id,"summary": CALENDAR_NAME})

# Create calendar event with date not equal to "April 6, 2025, at 11 AM" to match the case description
calendar_event = google_calendar.create_event(
    calendarId=calendar_id,
      resource={"summary": CALENDAR_EVENT_NAME,
      "location": CALENDAR_EVENT_LOCATION,
      "description": CALENDAR_EVENT_DESCRIPTION,
      "start": {
          "dateTime": CALENDAR_EVENT_START_DATETIME,
      },
      "end": {
          "dateTime": CALENDAR_EVENT_END_DATETIME,
      }})

print(f"Calendar event created: {calendar_event}")


# --- Autofix: standardize calendars' timezone to UTC ---
def set_all_calendars_timezone_to_utc():
    """
    Set timeZone='UTC' for every calendar, preserving summary/description.
    Skips calendars already in UTC.
    """
    cl = google_calendar.list_calendar_list_entries()
    items = cl.get("items", [])
    results = []

    for cal in items:
        cal_id = cal.get("id")
        summary = cal.get("summary", "")
        description = cal.get("description", "")

        # Only send fields allowed by UpdateCalendarInputResourceModel
        resource = {
            "summary": summary,
            "description": description,
            "timeZone": "UTC",
        }

        updated = google_calendar.update_calendar_metadata(
            calendarId=cal_id,
            resource=resource
        )
        results.append(updated)

    return results

# Apply after DBs are initiated
set_all_calendars_timezone_to_utc()

Project created: {'created': True, 'project': {'key': 'AGT', 'name': 'AGILE_TEAM', 'lead': None}}
Issue created: {'id': 'ISSUE-4', 'fields': {'project': 'AGT', 'summary': 'Sprint 12 Retrospective', 'issuetype': 'Task', 'description': '', 'priority': 'Low', 'status': 'In Progress', 'assignee': {'name': 'Unassigned'}, 'attachments': [], 'due_date': None, 'comments': [], 'created': '2025-10-18T11:43:21.100517', 'updated': '2025-10-18T11:43:21.100531', 'components': []}}
Calendar created: {'summary': 'Work', 'timeZone': 'UTC', 'id': '038938e3-ccc0-4e6d-a67c-da59cdbba3d8'}
Calendar event created: {'id': '09f207f8-a5e6-4135-bb5b-c3980d6fd3c6', 'summary': 'Sprint 12 Retrospective', 'description': 'Discussion about Sprint 12.', 'start': {'dateTime': '2025-03-31T11:00:00+00:00', 'timeZone': None}, 'end': {'dateTime': '2025-03-31T12:00:00+00:00', 'timeZone': None}, 'recurrence': None, 'attendees': None, 'reminders': None, 'location': 'Conference Room A', 'attachments': None, 'extendedProperties'

[{'id': 'cal-1000',
  'summary': 'Work Calendar',
  'description': 'Company-wide meetings and deadlines',
  'timeZone': 'UTC',
  'primary': True},
 {'id': 'cal-2000',
  'summary': 'Personal Calendar',
  'description': 'My personal events and reminders',
  'timeZone': 'UTC',
  'primary': False},
 {'id': 'cal-3000',
  'summary': 'Family Calendar',
  'description': 'Shared events with family members',
  'timeZone': 'UTC',
  'primary': False},
 {'id': 'cal-4000',
  'summary': 'Project Calendar',
  'description': '',
  'timeZone': 'UTC',
  'primary': False},
 {'id': 'cal-5000',
  'summary': 'Team Collaboration Calendar',
  'description': 'Shared calendar for team meetings and collaborative events',
  'timeZone': 'UTC',
  'primary': False},
 {'summary': 'Work',
  'timeZone': 'UTC',
  'id': '038938e3-ccc0-4e6d-a67c-da59cdbba3d8'}]

# Initial Assertion
1. Assert that exactly one Jira project named 'AGILE_TEAM' exists in the system.
2. Assert that the project 'AGILE_TEAM' contains exactly one jira issue with the summary 'Sprint 12 Retrospective' exists in the system.
3. Assert that the status of 'Sprint 12 Retrospective' has already been set to In Progress.
4. Assert that "Work" calendar exists.
5. Assert that "Work" calendar has exactly one event named "Sprint 12 Retrospective" in the system.
6. Assert that the event titled "Sprint 12 Retrospective" has not been scheduled to April 6, 2025, at 11 AM.

In [ ]:
from Scripts.assertions_utils import compare_strings, compare_datetimes
import google_calendar
import jira
from datetime import datetime, timezone

# ---------- Constants ----------
JIRA_ISSUE_SUMMARY = "Sprint 12 Retrospective"
JIRA_PROJECT_NAME = "AGILE_TEAM"
ISSUE_NEW_STATUS = "In Progress"
CALENDAR_EVENT_NAME = "Sprint 12 Retrospective"
CALENDAR_NAME = "Work"
CALENDAR_EVENT_NEW_DATE = "2025-04-06T11:00:00Z"  # target we assert the event is NOT scheduled to

# ---------- Helpers ----------
def to_utc(dt_str: str) -> datetime:
    """
    Parse ISO-8601 datetimes robustly:
    - Accepts 'Z', '+00:00', or any ?HH:MM offset
    - Accepts date-only (YYYY-MM-DD) by assuming 00:00:00
    - Returns timezone-aware UTC datetime
    """
    if not isinstance(dt_str, str) or not dt_str.strip():
        raise AssertionError("Invalid datetime string received for parsing.")
    s = dt_str.strip()
    if "T" not in s:  # all-day: YYYY-MM-DD
        s = s + "T00:00:00Z"
    # Normalize Z to +00:00 then parse; works for any offset
    dt = datetime.fromisoformat(s.replace("Z", "+00:00"))
    return dt.astimezone(timezone.utc)

# ---------- Jira Assertions ----------
# 1) Project exists exactly once
jira_projects = jira.get_all_projects()
agile_team_project = [
    p for p in jira_projects.get("projects", [])
    if compare_strings(p.get("name"), JIRA_PROJECT_NAME)
]
assert len(agile_team_project) == 1, f"Exactly one project named '{JIRA_PROJECT_NAME}' should exist."

# 2) Exactly one issue with the summary in that project
project_key = agile_team_project[0]["key"]
jql_query = f"project = '{project_key}' AND summary = '{JIRA_ISSUE_SUMMARY}'"
issues = jira.search_issues_jql(jql=jql_query).get("issues", [])
assert len(issues) == 1, f"Project '{JIRA_PROJECT_NAME}' should contain exactly one issue with summary '{JIRA_ISSUE_SUMMARY}'."

# 3) Issue status is 'In Progress'
issue_status = issues[0].get("fields", {}).get("status")
assert compare_strings(issue_status, ISSUE_NEW_STATUS), f"The status of '{JIRA_ISSUE_SUMMARY}' should be '{ISSUE_NEW_STATUS}'."

# ---------- Calendar Assertions ----------
# 4) "Work" calendar exists
cal_list = google_calendar.list_calendar_list_entries()
work_cal = next((c for c in cal_list.get("items", []) if compare_strings(c.get("summary", ""), CALENDAR_NAME)), None)
assert work_cal is not None, f"'{CALENDAR_NAME}' calendar not found."

# 5) Exactly one 'Sprint 12 Retrospective' event on that calendar
work_cal_id = work_cal.get("id")
events_resp = google_calendar.list_events(calendarId=work_cal_id)
events = events_resp.get("items", [])
sprint_events = [e for e in events if compare_strings(e.get("summary", ""), CALENDAR_EVENT_NAME)]
assert len(sprint_events) == 1, f"'{CALENDAR_NAME}' calendar should have exactly one event named '{CALENDAR_EVENT_NAME}'."

# 6) The event has NOT been scheduled to 2025-04-06T11:00:00Z
ev = sprint_events[0]
start_obj = ev.get("start", {}) or {}
start_str = start_obj.get("dateTime") or start_obj.get("date")

event_start_utc = to_utc(start_str)
target_dt_utc = to_utc(CALENDAR_EVENT_NEW_DATE)

# compare_datetimes handles aware datetimes; both are normalized to UTC above
assert not compare_datetimes(event_start_utc, target_dt_utc, "eq"), \
    f"The event '{CALENDAR_EVENT_NAME}' should not be scheduled to '{target_dt_utc.isoformat()}'."

# Action
- Update the Jira issue with summary "Sprint 12 Retrospective" in the project AGILE_TEAM and set status to In Progress

- Reschedule the Google Calendar event "Sprint 12 Retrospective" in "Work" calendar to April 6, 2025, at 11 AM.

In [ ]:
import gmail
import jira
import google_calendar
from datetime import timedelta, timezone
from dateutil import parser  # Handles ISO 8601 datetime strings with Z or +00:00 offsets

# Define global/context variables
PROJECT_NAME = "AGILE_TEAM"
ISSUE_SUMMARY = "Sprint 12 Retrospective"
STATUS = "In Progress"
CALENDAR_NAME = "Work"
EVENT_TITLE = "Sprint 12 Retrospective"
START_TIME = "2025-04-06T11:00:00Z"

# Search for the Jira issue based on project and summary
jira_projects = jira.get_all_projects()
agile_team_project = [project for project in jira_projects.get("projects", []) if project.get("name") == PROJECT_NAME]

jql_query=f"project = '{agile_team_project[0]['key']}' AND summary = '{ISSUE_SUMMARY}'"
agile_team_retrospective_issues = jira.search_issues_jql(jql=jql_query).get('issues',[])
issue_status = agile_team_retrospective_issues[0].get('fields', {}).get('status')

if issue_status == STATUS:
    print(f"Jira issue status is already set to '{STATUS}', skipping update.")
else:
    jira.update_issue_by_id(
        issue_id=agile_team_retrospective_issues[0].get('id'),
        fields={"status": STATUS}
    )
    print(f"Updated the status of the Jira issue '{ISSUE_SUMMARY}' to '{STATUS}'.")


# Calendar API action
calendar_lists = google_calendar.list_calendar_list_entries()
work_calendar = next((cal for cal in calendar_lists.get('items', [])
                      if cal.get('summary', '') == CALENDAR_NAME), None)

if work_calendar:
    work_calendar_id = work_calendar.get('id')

    # Check if the calendar has exactly one "Sprint 12 Retrospective" event
    work_calendar_events = google_calendar.list_events(calendarId=work_calendar_id)
    sprint_event = [event for event in work_calendar_events.get('items', [])
                    if event.get('summary') == EVENT_TITLE]

    if len(sprint_event) == 1:
        event_id = sprint_event[0].get('id')
        current_event = sprint_event[0]
        current_start_time = current_event.get('start', {}).get('dateTime')

        # Parse both current and expected start times into aware datetimes
        current_start_dt = parser.isoparse(current_start_time).astimezone(timezone.utc)
        expected_start_dt = parser.isoparse(START_TIME).astimezone(timezone.utc)

        if current_start_dt == expected_start_dt:
            print(f"Event '{EVENT_TITLE}' is already scheduled at {START_TIME}. No rescheduling needed.")
        else:
            # Get the current event data to preserve all existing properties
            existing_event = google_calendar.get_event(
                calendarId=work_calendar_id,
                eventId=event_id
            )

            # Calculate new end time (preserve the same duration)
            current_start = current_event.get('start', {}).get('dateTime')
            current_end = current_event.get('end', {}).get('dateTime')

            if current_start and current_end:
                start_dt = parser.isoparse(current_start)
                end_dt = parser.isoparse(current_end)
                duration = end_dt - start_dt

                new_start_dt = parser.isoparse(START_TIME)
                new_end_dt = new_start_dt + duration
                new_end_time = new_end_dt.isoformat()
            else:
                # Fallback: assume 1 hour duration
                new_start_dt = parser.isoparse(START_TIME)
                new_end_dt = new_start_dt + timedelta(hours=1)
                new_end_time = new_end_dt.isoformat()

            # Preserve all existing properties and only update start/end times
            updated_event_data = {
                "summary": existing_event.get('summary'),
                "description": existing_event.get('description'),
                "location": existing_event.get('location'),
                "start": {
                    "dateTime": new_start_dt.isoformat(),
                    "timeZone": existing_event.get('start', {}).get('timeZone')
                },
                "end": {
                    "dateTime": new_end_time,
                    "timeZone": existing_event.get('end', {}).get('timeZone')
                },
                "attendees": existing_event.get('attendees'),
                "reminders": existing_event.get('reminders'),
                "recurrence": existing_event.get('recurrence'),
                "extendedProperties": existing_event.get('extendedProperties')
            }

            # Remove None values to avoid overwriting existing data with null
            updated_event_data = {k: v for k, v in updated_event_data.items() if v is not None}

            google_calendar.update_event(
                calendarId=work_calendar_id,
                eventId=event_id,
                resource=updated_event_data
            )

            # Verify if the event has been rescheduled
            updated_event = google_calendar.get_event(
                calendarId=work_calendar_id,
                eventId=event_id
            )
            sprint_event_scheduled_start = parser.isoparse(updated_event.get('start', {}).get('dateTime')).astimezone(timezone.utc)

            if sprint_event_scheduled_start == expected_start_dt:
                print(f"Event '{EVENT_TITLE}' successfully rescheduled to {START_TIME}.")
            else:
                print(f"Failed to reschedule the event. Expected: {START_TIME}, Found: {sprint_event_scheduled_start.isoformat()}")

    elif len(sprint_event) == 0:
        print(f"No event named '{EVENT_TITLE}' found in the '{CALENDAR_NAME}' calendar.")
    else:
        print(f"Multiple events named '{EVENT_TITLE}' found in the '{CALENDAR_NAME}' calendar.")
else:
    print(f"Calendar '{CALENDAR_NAME}' not found.")


Jira issue status is already set to 'In Progress', skipping update.
Event 'Sprint 12 Retrospective' successfully rescheduled to 2025-04-06T11:00:00Z.


# Golden Answer
The Jira issue is already in the 'In Progress' status, so no update is required.

# Final Assertion
1. Assert that the Calendar event titled "Sprint 12 Retrospective" has been rescheduled to April 6, 2025, at 11 AM.

In [ ]:
from Scripts.assertions_utils import *
import google_calendar
from datetime import timezone
from dateutil import parser  # Robust ISO 8601 parser: handles Z, ±HH:MM, named TZ when provided

# Constants
CALENDAR_EVENT_NAME = "Sprint 12 Retrospective"
CALENDAR_NAME = "Work"
CALENDAR_EVENT_NEW_DATE = "2025-04-06T11:00:00Z"

# Safe initialization
work_calendar = None
actual_event = None
sprint_start_utc = None
expected_start_utc = None

# --- Discover calendar safely ---
try:
    calendar_lists = google_calendar.list_calendar_list_entries()
    items = (calendar_lists or {}).get('items', [])
    work_calendar = next((cal for cal in items if compare_strings(cal.get('summary', ''), CALENDAR_NAME)), None)
except Exception:
    work_calendar = None

# --- Find target event safely ---
if work_calendar and work_calendar.get('id'):
    try:
        work_calendar_id = work_calendar.get('id')
        events = google_calendar.list_events(calendarId=work_calendar_id)
        event_items = (events or {}).get('items', [])
        actual_event = next((e for e in event_items if compare_strings(e.get('summary'), CALENDAR_EVENT_NAME)), None)
    except Exception:
        actual_event = None
else:
    actual_event = None

# --- Parse times robustly and compare by instant ---
if actual_event:
    try:
        start_dict = (actual_event.get('start') or {})  # may contain 'dateTime' or 'date' (+ optional 'timeZone')
        start_time_str = start_dict.get('dateTime') or start_dict.get('date')

        if start_time_str:
            # parser.isoparse handles: '2025-04-06T11:00:00Z', '2025-04-06T07:00:00-04:00', etc.
            parsed_start = parser.isoparse(start_time_str)

            # If the start is date-only (all-day), compare on date; otherwise compare instants in UTC.
            if start_dict.get('date') is not None:
                # expected is a datetime; compare calendar date match after normalizing expected to UTC date
                expected_start_utc = parser.isoparse(CALENDAR_EVENT_NEW_DATE).astimezone(timezone.utc)
                sprint_start_utc = parsed_start  # date-only object; keep as-is
                # Convert date-only comparison inputs to dates
                same_day = (sprint_start_utc.date() == expected_start_utc.date())
                assert same_day, (
                    f"Event '{CALENDAR_EVENT_NAME}' not rescheduled to the expected date "
                    f"{expected_start_utc.date()} (found {sprint_start_utc.date()})."
                )
                # Short-circuit successful assertion path for all-day events
                # (No further strict checks like location/description/timezone syntax)
                pass
            else:
                # Time-aware (or naive) datetime: normalize to UTC for instant comparison.
                # If naive (no tzinfo), assume API-provided explicit 'timeZone' would be applied by Google;
                # parser will have tzinfo when string includes Z/±HH:MM; either way, coerce to UTC safely.
                if parsed_start.tzinfo is None:
                    # Treat as UTC if tzinfo missing; we DON'T fail the test on missing tz syntax.
                    parsed_start = parsed_start.replace(tzinfo=timezone.utc)
                sprint_start_utc = parsed_start.astimezone(timezone.utc)
                expected_start_utc = parser.isoparse(CALENDAR_EVENT_NEW_DATE).astimezone(timezone.utc)

        else:
            sprint_start_utc = None
            expected_start_utc = None
    except Exception:
        sprint_start_utc = None
        expected_start_utc = None

# --- Final assertion: only what actually proves a reschedule happened ---
assert all([
    work_calendar is not None,
    actual_event is not None,
    sprint_start_utc is not None,
    expected_start_utc is not None if expected_start_utc is not None else True  # handled in all-day case
]), f"Event '{CALENDAR_EVENT_NAME}' not found or could not parse dates."

# If we’re in the datetime (non all-day) path, verify instant equality in UTC.
# (For all-day path, equality was asserted above.)
if isinstance(sprint_start_utc, type(parser.isoparse("2025-01-01T00:00:00Z"))):  # datetime instance
    assert compare_datetimes(sprint_start_utc, expected_start_utc, "eq"), (
        f"Event '{CALENDAR_EVENT_NAME}' is not correctly rescheduled to {CALENDAR_EVENT_NEW_DATE} "
        f"(found {sprint_start_utc.isoformat()})."
    )

AssertionError: Event 'Sprint 12 Retrospective' is not correctly rescheduled to 2025-04-06T11:00:00Z.